# 🗄️ Notebook 05: Vector Stores

**LangChain 1.0.5+ | Mixed Level Class**

## 🎯 Objectives
1. Understand vector stores
2. Use InMemoryVectorStore (testing)
3. Use FAISS (production)
4. Use Chroma (persistent)
5. Compare vector stores

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
load_dotenv(override=True)
print("✅ Setup complete")

## 1. What are Vector Stores? 🤔

### 🔰 BEGINNER

**Vector Stores** are databases optimized for finding similar vectors.

Like a library:
- Traditional database: "Find book with ISBN 12345"
- Vector store: "Find books similar to this topic"

They enable **semantic search**!

## 2. InMemoryVectorStore (Simple)

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_core.documents import Document

# Initialize
embeddings = NVIDIAEmbeddings(model="nvidia/llama-nemotron-embed-1b-v2")
vector_store = InMemoryVectorStore(embedding=embeddings)

# Add documents
docs = [
    Document(page_content="Python is a programming language"),
    Document(page_content="Machine learning uses algorithms"),
    Document(page_content="The sun is a star")
]

vector_store.add_documents(docs)
print(f"✅ Added {len(docs)} documents\n")

# Search
results = vector_store.similarity_search("What is Python?", k=2)

print("Search Results:")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")

## 3. FAISS (Fast & Production-Ready)

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

# Load and split a PDF
pdf_path = "pdfs/rag.pdf"

if Path(pdf_path).exists():
    print("Loading PDF...")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    
    # Split
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = splitter.split_documents(documents)
    print(f"Split into {len(chunks)} chunks")
    
    # Create FAISS vectorstore
    print("Creating embeddings (this may take a minute)...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    
    # Save to disk
    vectorstore.save_local("./faiss_index_notebook")
    print("✅ FAISS index saved\n")
    
    # Search
    query = "What is RAG?"
    results = vectorstore.similarity_search(query, k=3)
    
    print(f"Query: {query}\n")
    for i, doc in enumerate(results, 1):
        print(f"Result {i}:")
        print(f"  {doc.page_content[:150]}...\n")
else:
    print(f"❌ PDF not found: {pdf_path}")

In [ ]:
query = "What is RAG-Sequence Model?"
results = vectorstore.similarity_search(query, k=3)
results

## 4. Loading Existing FAISS Index

In [ ]:
# Load existing index
if Path("./faiss_index_notebook").exists():
    loaded_vectorstore = FAISS.load_local(
        "./faiss_index_notebook",
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("✅ Loaded existing FAISS index")
    
    # Use it
    results = loaded_vectorstore.similarity_search("transformers", k=2)
    print(f"\nFound {len(results)} results")
else:
    print("No existing index found")

## 5. Chroma (Persistent & Easy)

In [ ]:
!uv pip install langchain-chroma chromadb

In [ ]:
from langchain_chroma import Chroma

In [ ]:
from langchain_chroma import Chroma

# Create Chroma vectorstore
chroma_store = Chroma(
    collection_name="my_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_notebook"
)

# Add documents
sample_docs = [
    Document(
        page_content="RAG combines retrieval and generation",
        metadata={"topic": "rag", "difficulty": "intermediate"}
    ),
    Document(
        page_content="LangChain simplifies LLM applications",
        metadata={"topic": "langchain", "difficulty": "beginner"}
    )
]

chroma_store.add_documents(sample_docs)
print("✅ Added documents to Chroma\n")

# Search with metadata filter
results = chroma_store.similarity_search(
    "Tell me about RAG",
    k=2,
    filter={"topic": "rag"}
)

print("Filtered search results:")
for doc in results:
    print(f"  {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")

## 6. Vector Store Comparison

| Store | Persistent | Speed | Metadata Filtering | Best For |
|-------|-----------|-------|-------------------|----------|
| **InMemory** | ❌ | ⚡⚡⚡ | ❌ | Testing |
| **FAISS** | ✅ | ⚡⚡⚡ | Limited | Production, Speed |
| **Chroma** | ✅ | ⚡⚡ | ✅ | Development, Filtering |

## Summary

✅ Vector stores enable semantic search
✅ InMemory for testing
✅ FAISS for speed and scale
✅ Chroma for metadata filtering
✅ Always persist for production!

**Next:** Retrieval Strategies!